In [1]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import torch
torch.set_default_device("cuda")
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

from flow_model import HCDFlowResMLP, HCDFlow
from metrics import sa,l1, l2, pcc

# Load pretrain model

In [2]:
model_path = r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\models\pep=256\blocks=12\model_6e_1024b.pth"
model = HCDFlowResMLP(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128, num_blocks=12)
model.load_state_dict(torch.load(model_path))

model.eval()

HCDFlowResMLP(
  (cond_embedding): ConditionEmbedding(
    (pep_embedding): Embedding(22, 256, padding_idx=0)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (mlp): ResMLPWithConditioning(
    (blocks): ModuleList(
      (0-11): 12 x ResBlock(
        (norm): RMSNorm((174,), eps=None, elementwise_a

# Test on test data

In [3]:
train_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\holdout_hcd.hdf5"

In [4]:
with h5py.File(train_path, "r") as f:
    print("Keys:", list(f.keys()))
    # total_samples = f["sequence_integer"].shape[0]
    # random_idx = np.random.choice(total_samples, size=16, replace=False)
    # random_idx.sort()
    # seqs = f["sequence_integer"][random_idx]
    # charges_oh = f["precursor_charge_onehot"][random_idx]
    # intensities = f["intensities_raw"][random_idx]
    seqs = f["sequence_integer"][:]
    charges_oh = f["precursor_charge_onehot"][:]
    intensities = f["intensities_raw"][:]    

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer'
]

In [5]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

In [6]:
# charges

In [7]:
seq_tensors = torch.tensor(seqs, device="cpu") 
charge_tensors = torch.tensor(charges, device="cpu")
intensity_tensors = torch.tensor(intensities, device="cpu")

dataset = TensorDataset(seq_tensors, charge_tensors, intensity_tensors)


In [8]:
intensity_tensors = torch.tensor(intensities, dtype=torch.float32)
seq_tensors = torch.tensor(seqs, dtype=torch.long)
charge_tensors = torch.tensor(charges, dtype=torch.float32)

In [9]:
batch = 16
total_sample = len(seqs)
total_test = 1

loader = DataLoader(
    dataset,
    batch_size=batch,
    shuffle=False,
    pin_memory=True,
    num_workers=4
)

noises = torch.randn(total_test, intensities.shape[1]).unsqueeze(1).expand(total_test, batch, -1)

In [10]:
for noise in noises:
    print(noise.shape)

torch.Size([16, 174])

In [11]:
sa_losses   = []
pcc_losses  = []
with torch.no_grad():
    device = torch.get_default_device()
    for seq_b, charge_b, intensity_b in loader:
        seq_b = seq_b.to(device, non_blocking=True)
        charge_b = charge_b.to(device, non_blocking=True)
        intensity_b = intensity_b.to(device, non_blocking=True)
        
        # print(seq_b.shape, charge_b.shape, intensity_b.shape)
        for noise in noises:
            # print(noise.shape)
            generated = model.sample(noise, seq_b, charge_b)
            
            sa_losses.append(sa(intensity_b, generated))
            pcc_losses.append(pcc(intensity_b, generated))

KeyboardInterrupt: 

In [ ]:
sum(sa_losses[0])/len(sa_losses)

(0.37836835677116687, 0.40122327693138826)

In [19]:
sum([m for m, ma in pcc_losses]) / len(pcc_losses)

0.8215308079434386

# Test on test set